# 🚗 Vehicle Insurance Cross-Sell Analysis — Exploratory Data Analysis (EDA)

**A Beginner-Friendly Data Analytics Portfolio Project**

### 📌 Project Objective
An insurance company wants to know which of its existing **health insurance customers** are likely to also say **"yes"** to a **vehicle insurance** offer. This project explores the customer dataset to understand **who responds positively** and **why**, using only Exploratory Data Analysis (EDA) — no Machine Learning.

### 🛠️ Tools & Technologies
- **Python** – programming language
- **Pandas** – data loading, cleaning, and analysis
- **NumPy** – numerical operations
- **Matplotlib** & **Seaborn** – data visualization
- **Google Colab / Jupyter Notebook** – environment

### 📂 Dataset Description
The dataset (`Vehicle_Insurance.csv`) contains **381,109 rows** and **12 columns** of customer records:

| Column | Meaning |
|---|---|
| id | Unique customer ID |
| Gender | Male / Female |
| Age | Customer's age |
| Driving_License | 1 = has license, 0 = no license |
| Region_Code | Code for the customer's region |
| Previously_Insured | 1 = already has vehicle insurance, 0 = does not |
| Vehicle_Age | Age of the customer's vehicle |
| Vehicle_Damage | Whether the vehicle was damaged in the past |
| Annual_Premium | Amount paid yearly for health insurance |
| Policy_Sales_Channel | Code for the channel used to contact the customer |
| Vintage | Number of days the customer has been associated with the company |
| Response | **Target** — 1 = interested in vehicle insurance, 0 = not interested |

> 📝 **Note:** This dataset does not contain actual calendar dates, so "date/time analysis" in this project is done using the **Vintage** column (days associated with the company), which is treated as a time-based feature.

### 🧭 Project Structure
1. Import Libraries
2. Load & Understand the Data
3. Data Cleaning
4. Feature Engineering
5. GroupBy Analysis
6. Sorting & Filtering
7. Visualizations (14 charts)
8. Business Insights
9. Conclusion


## 1️⃣ Import Libraries
We start by importing the libraries we need for data handling (Pandas, NumPy) and visualization (Matplotlib, Seaborn).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Makes charts look clean and readable
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)


## 2️⃣ Load & Understand the Data
If you are running this in **Google Colab**, first upload `Vehicle_Insurance.csv` using the file upload button on the left panel, then run the cell below.

In [ ]:
df = pd.read_csv('Vehicle_Insurance.csv')

# Look at the first 5 rows
df.head()


**Checking the shape (rows, columns), column names, and data types.**

In [ ]:
print("Shape of dataset (rows, columns):", df.shape)
print("\nColumn names:\n", df.columns.tolist())
print("\nData types:\n")
df.dtypes


In [ ]:
# Statistical summary of numeric columns
df.describe()


This chart is not a visualization — it's a quick summary table. It tells us the average, minimum, maximum, and spread of numeric columns like `Age`, `Annual_Premium`, and `Vintage`.

## 3️⃣ Data Cleaning
Every good analysis starts with clean data. We check for missing values and duplicate rows.

In [ ]:
# Check for missing values in each column
df.isnull().sum()


**Result:** This dataset has **no missing values**, so no imputation is needed here. (In real-world datasets, you would fill missing numeric values with `median()` and missing categorical values with `mode()`.)

In [ ]:
# Check for duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

# Remove duplicates (if any) and keep the data clean
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)


In [ ]:
# Quick sanity check on categorical columns
print(df['Gender'].unique())
print(df['Vehicle_Age'].unique())
print(df['Vehicle_Damage'].unique())


The categorical columns are already clean and consistent (no spelling mistakes or extra spaces), so no further text-cleaning is required.

## 4️⃣ Feature Engineering (Creating Useful Columns)
We create a few new columns to make the analysis easier and more insightful.

In [ ]:
# 4.1 Age Group — group customers into readable age buckets
def age_group(age):
    if age <= 30:
        return '20-30'
    elif age <= 40:
        return '31-40'
    elif age <= 50:
        return '41-50'
    elif age <= 60:
        return '51-60'
    else:
        return '61+'

df['Age_Group'] = df['Age'].apply(age_group)
df['Age_Group'].value_counts()


In [ ]:
# 4.2 Tenure Group — using 'Vintage' (days with company) as our time-based feature
def tenure_group(days):
    if days <= 100:
        return '0-100 days'
    elif days <= 200:
        return '101-200 days'
    else:
        return '201-300 days'

df['Tenure_Group'] = df['Vintage'].apply(tenure_group)
df['Tenure_Group'].value_counts()


In [ ]:
# 4.3 Premium Category — group Annual Premium into Low / Medium / High
def premium_category(premium):
    if premium <= 30000:
        return 'Low'
    elif premium <= 40000:
        return 'Medium'
    else:
        return 'High'

df['Premium_Category'] = df['Annual_Premium'].apply(premium_category)
df['Premium_Category'].value_counts()


We now have three new, easy-to-read columns: **Age_Group**, **Tenure_Group** (our date/time-based feature), and **Premium_Category**.

## 5️⃣ GroupBy Analysis
Here we group the data by different categories to find patterns in customer **Response** (interest in vehicle insurance).

In [ ]:
# Overall response rate
overall_rate = df['Response'].mean() * 100
print(f"Overall response rate: {overall_rate:.2f}%")


In [ ]:
# Response rate by Gender
gender_response = (df.groupby('Gender')['Response'].mean() * 100).round(2)
gender_response


In [ ]:
# Response rate by Age Group
age_response = (df.groupby('Age_Group')['Response'].mean() * 100).round(2)
age_response


In [ ]:
# Response rate by Vehicle Age
vehicle_age_response = (df.groupby('Vehicle_Age')['Response'].mean() * 100).round(2)
vehicle_age_response


In [ ]:
# Response rate by Vehicle Damage history
damage_response = (df.groupby('Vehicle_Damage')['Response'].mean() * 100).round(2)
damage_response


In [ ]:
# Response rate by Previously Insured status
insured_response = (df.groupby('Previously_Insured')['Response'].mean() * 100).round(2)
insured_response


In [ ]:
# Average Annual Premium by Vehicle Age
premium_by_vehicle_age = df.groupby('Vehicle_Age')['Annual_Premium'].mean().round(2)
premium_by_vehicle_age


In [ ]:
# Response rate by Tenure Group (our date/time-based analysis)
tenure_response = (df.groupby('Tenure_Group')['Response'].mean() * 100).round(2)
tenure_response


In [ ]:
# Top 10 regions by response rate (only regions with 500+ customers, to keep it reliable)
region_stats = df.groupby('Region_Code').agg(
    Total_Customers=('id', 'count'),
    Response_Rate=('Response', 'mean')
)
region_stats['Response_Rate'] = (region_stats['Response_Rate'] * 100).round(2)
region_stats = region_stats[region_stats['Total_Customers'] >= 500]
top_regions = region_stats.sort_values('Response_Rate', ascending=False).head(10)
top_regions


## 6️⃣ Sorting & Filtering
We now sort and filter the data to answer specific business questions.

In [ ]:
# Sort: Top 10 customers with the highest Annual Premium
top_premium_customers = df.sort_values('Annual_Premium', ascending=False)[
    ['id', 'Age', 'Gender', 'Annual_Premium', 'Response']
].head(10)
top_premium_customers


In [ ]:
# Filter: High-potential customers -> vehicle was damaged AND not previously insured
high_potential = df[(df['Vehicle_Damage'] == 'Yes') & (df['Previously_Insured'] == 0)]

print("Number of high-potential customers:", len(high_potential))
print(f"Their response rate: {high_potential['Response'].mean()*100:.2f}%")


This filtered segment shows a **much higher response rate** than the overall average — a useful signal for targeting future marketing campaigns.

## 7️⃣ Visualizations
Below are 14 beginner-friendly charts that visually explain the patterns found above.

**Chart 1 — Gender Distribution.** This chart shows how many male and female customers are in the dataset.

In [ ]:
plt.figure(figsize=(6,5))
sns.countplot(data=df, x='Gender', hue='Gender', palette='pastel', legend=False)
plt.title('Customer Count by Gender', fontsize=14)
plt.xlabel('Gender')
plt.ylabel('Number of Customers')
plt.show()


**Chart 2 — Age Distribution.** This histogram shows the age range of customers and where most customers fall.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Age'], bins=30, color='skyblue', kde=True)
plt.title('Distribution of Customer Age', fontsize=14)
plt.xlabel('Age')
plt.ylabel('Number of Customers')
plt.show()


**Chart 3 — Vehicle Age Distribution.** This chart shows how many customers own vehicles of each age category.

In [ ]:
plt.figure(figsize=(6,5))
order = ['< 1 Year', '1-2 Year', '> 2 Years']
sns.countplot(data=df, x='Vehicle_Age', hue='Vehicle_Age', order=order, palette='Set2', legend=False)
plt.title('Customer Count by Vehicle Age', fontsize=14)
plt.xlabel('Vehicle Age')
plt.ylabel('Number of Customers')
plt.show()


**Chart 4 — Vehicle Damage History.** This chart shows how many customers had a previously damaged vehicle.

In [ ]:
plt.figure(figsize=(6,5))
sns.countplot(data=df, x='Vehicle_Damage', hue='Vehicle_Damage', palette='Set1', legend=False)
plt.title('Vehicle Damage History of Customers', fontsize=14)
plt.xlabel('Vehicle Damage')
plt.ylabel('Number of Customers')
plt.show()


**Chart 5 — Target Variable: Response.** This chart shows how many customers are interested (1) vs not interested (0) in vehicle insurance.

In [ ]:
plt.figure(figsize=(6,5))
ax = sns.countplot(data=df, x='Response', hue='Response', palette='coolwarm', legend=False)
plt.title('Customer Response to Vehicle Insurance Offer', fontsize=14)
plt.xlabel('Response (0 = Not Interested, 1 = Interested)')
plt.ylabel('Number of Customers')
plt.show()


**Chart 6 — Response Rate by Gender.** This bar chart compares how likely male vs female customers are to say yes.

In [ ]:
plt.figure(figsize=(6,5))
gender_response.plot(kind='bar', color=['#ff9999','#66b3ff'])
plt.title('Response Rate by Gender', fontsize=14)
plt.xlabel('Gender')
plt.ylabel('Response Rate (%)')
plt.xticks(rotation=0)
plt.show()


**Chart 7 — Response Rate by Age Group.** This chart shows which age groups are most interested in vehicle insurance.

In [ ]:
plt.figure(figsize=(7,5))
age_response.plot(kind='bar', color='mediumseagreen')
plt.title('Response Rate by Age Group', fontsize=14)
plt.xlabel('Age Group')
plt.ylabel('Response Rate (%)')
plt.xticks(rotation=0)
plt.show()


**Chart 8 — Response Rate by Vehicle Age.** This chart shows that customers with older vehicles respond very differently than those with newer ones.

In [ ]:
plt.figure(figsize=(6,5))
vehicle_age_response.loc[order].plot(kind='bar', color='orange')
plt.title('Response Rate by Vehicle Age', fontsize=14)
plt.xlabel('Vehicle Age')
plt.ylabel('Response Rate (%)')
plt.xticks(rotation=0)
plt.show()


**Chart 9 — Response Rate by Vehicle Damage.** This chart shows the strong link between past vehicle damage and interest in insurance.

In [ ]:
plt.figure(figsize=(6,5))
damage_response.plot(kind='bar', color=['#8e44ad','#f39c12'])
plt.title('Response Rate by Vehicle Damage History', fontsize=14)
plt.xlabel('Vehicle Damage')
plt.ylabel('Response Rate (%)')
plt.xticks(rotation=0)
plt.show()


**Chart 10 — Response Rate by Previously Insured Status.** This chart shows that customers who already have insurance almost never respond positively.

In [ ]:
plt.figure(figsize=(6,5))
insured_response.plot(kind='bar', color=['#e74c3c','#2ecc71'])
plt.title('Response Rate: Previously Insured vs Not Insured', fontsize=14)
plt.xlabel('Previously Insured (0 = No, 1 = Yes)')
plt.ylabel('Response Rate (%)')
plt.xticks(rotation=0)
plt.show()


**Chart 11 — Annual Premium Distribution.** This histogram shows how much customers typically pay for their premium (values above the 99th percentile are removed just for a clearer chart).

In [ ]:
premium_capped = df[df['Annual_Premium'] <= df['Annual_Premium'].quantile(0.99)]

plt.figure(figsize=(8,5))
sns.histplot(premium_capped['Annual_Premium'], bins=40, color='teal')
plt.title('Distribution of Annual Premium', fontsize=14)
plt.xlabel('Annual Premium')
plt.ylabel('Number of Customers')
plt.show()


**Chart 12 — Average Annual Premium by Vehicle Age.** This chart compares how much customers with different vehicle ages typically pay.

In [ ]:
plt.figure(figsize=(6,5))
premium_by_vehicle_age.loc[order].plot(kind='bar', color='steelblue')
plt.title('Average Annual Premium by Vehicle Age', fontsize=14)
plt.xlabel('Vehicle Age')
plt.ylabel('Average Annual Premium')
plt.xticks(rotation=0)
plt.show()


**Chart 13 — Response Rate by Tenure Group (Time-Based Analysis).** This chart checks whether customers who have been with the company longer respond differently.

In [ ]:
plt.figure(figsize=(6,5))
tenure_order = ['0-100 days', '101-200 days', '201-300 days']
tenure_response.loc[tenure_order].plot(kind='bar', color='salmon')
plt.title('Response Rate by Customer Tenure (Vintage)', fontsize=14)
plt.xlabel('Days Associated with Company')
plt.ylabel('Response Rate (%)')
plt.xticks(rotation=0)
plt.show()


**Chart 14 — Top 10 Regions by Response Rate.** This chart highlights which regions have the highest interest in vehicle insurance (useful for targeted marketing).

In [ ]:
plt.figure(figsize=(9,5))
top_regions['Response_Rate'].plot(kind='bar', color='darkcyan')
plt.title('Top 10 Regions by Response Rate', fontsize=14)
plt.xlabel('Region Code')
plt.ylabel('Response Rate (%)')
plt.xticks(rotation=0)
plt.show()


**Chart 15 — Correlation Heatmap.** This chart shows how numeric columns relate to each other, including which factors have the strongest link with `Response`.

In [ ]:
plt.figure(figsize=(8,6))
numeric_cols = ['Age', 'Driving_License', 'Previously_Insured', 'Annual_Premium', 'Vintage', 'Response']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Numeric Features', fontsize=14)
plt.show()


## 8️⃣ Business Insights

1. **Overall response rate is ~12.3%** — only about 1 in 8 existing customers are currently interested in vehicle insurance, so targeting the right segment matters a lot.
2. **Vehicle damage history is the strongest signal** — customers with a damaged vehicle respond at **~23.8%**, compared to just **~0.5%** for those with no damage history.
3. **Customers who already have vehicle insurance almost never respond** (~0.09%) — it makes sense not to spend marketing budget on this group.
4. **Older vehicles show more interest** — customers with vehicles older than 2 years respond at **~29.4%**, much higher than customers with vehicles under 1 year old (**~4.4%**).
5. **Middle-aged customers (31–50) are the most interested age group**, with response rates around **21%**, compared to only **~4.5%** for the 20–30 age group.
6. **Male customers respond slightly more often** than female customers (**13.8% vs 10.4%**).
7. **How long a customer has been with the company (Vintage/Tenure) barely affects their interest** — response rate stays flat at around **12.3%** regardless of tenure group, so tenure alone is not a useful targeting factor.
8. **A combined segment of "damaged vehicle + not previously insured" customers responds at ~25%**, roughly **double the overall average** — this is a strong, easy-to-identify group to prioritize in campaigns.
9. **A small number of regions (e.g., Region 28) contain a very large share of customers** and also show above-average response rates, making them high-value regions for the sales team.
10. **Annual Premium and Vintage show very weak correlation with Response**, meaning premium amount and customer tenure alone don't predict interest — behavioral factors (damage history, previous insurance status, vehicle age) matter far more than financial or time-based ones.


## 9️⃣ Conclusion

This project explored a vehicle insurance cross-sell dataset of over **380,000 customers** using simple, beginner-friendly EDA techniques in Python. Without using any Machine Learning, we were able to clearly identify **who is most likely to be interested in vehicle insurance**:

- Customers whose vehicle was **previously damaged**
- Customers who are **not already insured**
- Customers with **older vehicles (2+ years)**
- Customers in the **31–50 age range**

On the other hand, customers who are **already insured**, own a **new vehicle**, or fall in the **20–30 age group** show very little interest.

These insights can directly help a marketing or sales team **prioritize outreach**, save on campaign costs, and **focus on the customer segments most likely to convert** — showing how basic data analysis, on its own, can support real business decisions.

---
### 👩‍💻 About this Project
This project was built as part of a Data Analytics portfolio to demonstrate skills in **Python, Pandas, NumPy, Matplotlib, and Seaborn** for real-world Exploratory Data Analysis.
